# Análisis diferencial de Keccak dinámico mediante MILP

**Objetivo:** Determinar el número mínimo de cajas‑S activas en la variante de Keccak con rondas dinámicas (R = 1..10, z = 4, 8) usando un modelo MILP con Convex Hull y estrategia de decisión.

**Resultados clave:** El mínimo es exactamente **R** para todos los casos, certificado por HiGHS en segundos.

Este cuaderno reproduce los experimentos y muestra los resultados en tabla.

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

from src.milp_keccak import analizar_con_decision
from src.experimentos import tabla
import json
import time

In [ ]:
# Ejecutar todos los casos (esto puede tardar ~2 minutos con 4 workers)
# Si quieres ejecutar solo algunos, modifica los rangos.
zs = [4, 8]
Rs = list(range(1, 11))
resultados = []

for z in zs:
    for R in Rs:
        print(f"Ejecutando z={z}, R={R}...")
        d = analizar_con_decision(R, z, limite_tiempo=60)
        resultados.append(d)

print("\nTodos los casos completados.")

In [ ]:
# Mostrar tabla
tabla(resultados)

In [ ]:
# Guardar resultados en JSON
with open('../resultados.json', 'w') as f:
    json.dump(resultados, f, indent=2)
print("Resultados guardados en ../resultados.json")

## Análisis de complejidad

La tabla anterior ya muestra la probabilidad y pares necesarios. Aquí se visualiza el crecimiento.

In [ ]:
import matplotlib.pyplot as plt

for z in zs:
    datos_z = [d for d in resultados if d['z'] == z]
    Rs_z = [d['R'] for d in datos_z]
    pares = [d['pares_log2'] for d in datos_z]
    plt.plot(Rs_z, pares, marker='o', label=f'z={z}')

plt.xlabel('Rondas (R)')
plt.ylabel('Log₂(pares necesarios)')
plt.title('Complejidad de datos para ataque diferencial')
plt.grid(True)
plt.legend()
plt.show()

**Conclusión:** El mínimo de cajas activas es exactamente R, lo que implica una probabilidad de `2⁻²ᴿ` y pares `2²ᴿ`. La variable dinámica endurece la primitiva exponencialmente con R.